# Qwen3-4B exp009 — đánh giá ABSA thực tế (Shopee)

**Mô hình:** `qwen3_4b_exp009` — Qwen3-4B-Instruct-2507 lượng hoá 4-bit, prompt CoT 1-shot (`qwen3-4b-instruct-2507/prompt-cot/exp009`, lượt L1).

**Vai trò:** mô hình LLM đại diện, đối chiếu với mô hình encoder trên cùng bộ dữ liệu thực tế.

**Chế độ:** chỉ suy luận — notebook này **không huấn luyện**, không cập nhật trọng số, không dùng để chọn mô hình hay chỉnh siêu tham số. Prompt CoT giữ nguyên như lượt L1.

**Dữ liệu:** 60 đánh giá son môi trên Shopee (tiếng Việt), cột `text` + 7 khía cạnh `stayingpower`, `texture`, `smell`, `price`, `colour`, `shipping`, `packing`. Nhãn mỗi khía cạnh là 0/1/2; **không có nhãn 3 (trung tính)**. `text` được đưa nguyên văn vào mô hình, không làm sạch, không chuẩn hoá, không tách từ.

**Trọng số trên Drive** (`<Drive>/SentimentX/models/`):
- `Qwen3-4B-Instruct-2507/` — **bắt buộc**, gồm `config.json`, `tokenizer.json`, `tokenizer_config.json`, `vocab.json`, `merges.txt`, `generation_config.json`, `model.safetensors.index.json` và các shard `.safetensors`.

**Dừng khi thiếu:** nếu không tìm thấy thư mục mô hình bắt buộc, ô chạy dừng bằng `FileNotFoundError` nêu rõ các đường dẫn đã thử và file còn thiếu; không có kết quả nào được ghi.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/TrieuKhac-dev/SentimentX"
REPO_BRANCH = "main"
DRIVE_ROOT_ENV = os.environ.get("DRIVE_ROOT", "").strip()


def _in_colab():
    try:
        import google.colab
        return True
    except Exception:
        return False


IN_COLAB = _in_colab()

if IN_COLAB:
    from google.colab import drive

    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive")

    DRIVE_ROOT = Path(DRIVE_ROOT_ENV) if DRIVE_ROOT_ENV else Path("/content/drive/MyDrive/SentimentX")
    CODE_DIR = Path("/content/SentimentX")

    if not (CODE_DIR / ".git").exists():
        subprocess.run(
            ["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(CODE_DIR)],
            check=True,
        )
    else:
        subprocess.run(
            ["git", "-C", str(CODE_DIR), "pull", "--ff-only", "origin", REPO_BRANCH],
            check=True,
        )

    os.environ["HF_HOME"] = "/content/hf_cache"
    os.environ["TRANSFORMERS_CACHE"] = "/content/hf_cache"
    Path("/content/hf_cache").mkdir(parents=True, exist_ok=True)

    subprocess.run(
        [
            sys.executable, "-m", "pip", "install", "-q",
            "-r", str(CODE_DIR / "requirements-serving.txt"),
            "peft", "accelerate", "bitsandbytes", "sentencepiece",
        ],
        check=True,
    )
else:
    here = Path.cwd().resolve()
    CODE_DIR = next(
        (path for path in [here, *here.parents] if (path / "serving" / "config.py").is_file()),
        None,
    )
    if CODE_DIR is None:
        raise FileNotFoundError("Không tìm thấy gốc repo (thiếu serving/config.py) từ " + str(here))
    DRIVE_ROOT = Path(DRIVE_ROOT_ENV) if DRIVE_ROOT_ENV else None

if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

print("Colab:", IN_COLAB)
print("CODE_DIR:", CODE_DIR)
print("DRIVE_ROOT:", DRIVE_ROOT)
print("HF_HOME:", os.environ.get("HF_HOME", "(chưa đặt)"))
print("TRANSFORMERS_CACHE:", os.environ.get("TRANSFORMERS_CACHE", "(chưa đặt)"))


In [ ]:
DRIVE_CSV = (DRIVE_ROOT / "data/raw/realworld/v0.1.0/shoppe_absa.csv") if DRIVE_ROOT else None
REPO_CSV = CODE_DIR / "notebooks/realworld/data/shoppe_absa.csv"

if DRIVE_CSV is not None and DRIVE_CSV.is_file():
    DATA_PATH = DRIVE_CSV
elif REPO_CSV.is_file():
    DATA_PATH = REPO_CSV
else:
    tried = [str(path) for path in (DRIVE_CSV, REPO_CSV) if path is not None]
    raise FileNotFoundError("Không tìm thấy shoppe_absa.csv. Đã thử: " + ", ".join(tried))

print("DATA_PATH:", DATA_PATH)
print("Kích thước:", DATA_PATH.stat().st_size, "byte")


In [ ]:
sys.path.insert(0, str(CODE_DIR / "notebooks/realworld"))

from eval_realworld import run_experiment

print("Đã nạp run_experiment từ", CODE_DIR / "notebooks/realworld/eval_realworld.py")


In [ ]:
MODEL_ID = "qwen3_4b_exp009"
CHUNK = 1
OUT_ROOT = (DRIVE_ROOT / "experiments/realworld") if IN_COLAB else (CODE_DIR / "notebooks/realworld/results")

result = run_experiment(
    model_id=MODEL_ID,
    data_path=DATA_PATH,
    out_root=OUT_ROOT,
    drive_root=DRIVE_ROOT,
    code_dir=CODE_DIR,
    chunk=CHUNK,
)


In [ ]:
import json

import pandas as pd

out_dir = Path(result["out_dir"])
metrics = json.loads((out_dir / "metrics.json").read_text(encoding="utf-8"))
accuracy = metrics["scores"]["accuracy"]
detection = metrics["scores"]["aspect_detection"]

print("Thư mục kết quả:", out_dir)
print("Accuracy: macro =", accuracy["macro"], "| micro =", accuracy["micro"], "| số ô =", accuracy["cells"])
print("Nhận diện khía cạnh: F1 macro =", detection["macro"]["f1"], "| F1 micro =", detection["micro"]["f1"])
print("Số dòng lỗi:", result["n_errors"])
pd.read_csv(result["predictions"], encoding="utf-8-sig").head()
